# PARC2026 FastWAM Colab gate notebook

明日の提出候補を作る前に、FastWAMが **Colab GPU / 10秒 / 24GB / 20GB ZIP** の条件へ入るかを順番に確認します。上から実行し、赤いゲートが出たら後続の重いセルを実行しません。

このNotebookの初版は (1) 依存確認、(2) 12GB checkpoint取得、(3) PARC観測変換、(4) 単発推論、(5) VRAM・速度計測、(6) 小型residual adapterの学習、までを扱います。公開重みだけの提出は行いません。

In [ ]:
# 0. 実験設定（最初に編集）
from pathlib import Path
import os, json, time, platform, subprocess

RUN_ID = 'F1-WAM-001'
MODEL_ID = 'lerobot/fastwam_libero_uncond_2cam224'
INFERENCE_STEPS = 10
N_ACTION_STEPS = 8       # 32個すべてを開ループ実行しない
INFERENCE_SEED = 42
# 大容量物はGoogle Driveへ保存しない。すべてColab VMの一時領域に置く。
HF_BACKUP_REPO = 'a2taka-sourse/parc2026-fastwam-runs'  # 必ずprivateで作る
WORK_ROOT = Path('/content/parc2026_fastwam')
MODEL_ROOT = WORK_ROOT / 'model_weights'
RESULT_ROOT = WORK_ROOT / 'experiments' / RUN_ID
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print({'run_id': RUN_ID, 'model': MODEL_ID, 'result_root': str(RESULT_ROOT)})

In [ ]:
# 1. 切断復帰先を準備。Colab Secretsに HF_TOKEN を登録してから実行します。
# ベース重み・dataset cache・動画はアップロードせず、adapter / optimizer / rng / log / manifestだけ保存します。
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Colab SecretsへHF_TOKENを登録してください'
hf_api = HfApi(token=HF_TOKEN)
hf_api.create_repo(HF_BACKUP_REPO, private=True, exist_ok=True)
print(f'切断復帰先: {HF_BACKUP_REPO} (private)')

In [ ]:
# 2. GPUゲート。T4 16GBではロード失敗の可能性が高いです。
import torch
assert torch.cuda.is_available(), 'GPUランタイムへ変更してください'
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
runtime = {
    'gpu': gpu.name, 'vram_gib': round(gpu_gib, 2),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'python': platform.python_version(),
}
print(json.dumps(runtime, indent=2, ensure_ascii=False))
if gpu_gib < 22:
    print('WARNING: 22GiB未満です。取得・容量監査はできますが、モデルロードはOOMの可能性が高いです。')

In [ ]:
# 3. 実験環境。提出物ではGit依存を使わず、検証後にPyPI pinまたはvendorを確定します。
%pip install -q --upgrade 'lerobot[fastwam,dataset]==0.6.1' 'huggingface_hub>=0.34' scipy safetensors
import importlib.util
assert importlib.util.find_spec('lerobot.policies.fastwam') is not None, 'lerobot 0.6.1にFastWAMがありません'
print('FastWAM import: PASS')

In [ ]:
# 4. checkpoint取得。約12GBに加え、Wan VAE / UMT5関連資産が必要になる可能性があります。
from huggingface_hub import snapshot_download
WORK_ROOT.mkdir(parents=True, exist_ok=True)
local_model = Path(snapshot_download(MODEL_ID, local_dir=MODEL_ROOT))
files = sorted((p.relative_to(local_model).as_posix(), p.stat().st_size) for p in local_model.rglob('*') if p.is_file())
total_gib = sum(s for _, s in files) / 2**30
print(f'model snapshot: {total_gib:.2f} GiB')
for name, size in sorted(files, key=lambda x: x[1], reverse=True)[:15]:
    print(f'{size/2**30:7.2f} GiB  {name}')
if total_gib >= 19:
    raise RuntimeError('GATE FAIL: モデル資産だけで20GB提出上限が危険です')

In [ ]:
# 5. モデルと保存済みpre/postprocessorをロード
import gc, torch
from lerobot.policies.fastwam.modeling_fastwam import FastWAMPolicy
from lerobot.processor import PolicyProcessorPipeline

device = torch.device('cuda')
torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
policy = FastWAMPolicy.from_pretrained(local_model, strict=False).to(device).eval()
policy.config.num_inference_steps = INFERENCE_STEPS
policy.config.n_action_steps = N_ACTION_STEPS
policy.config.inference_seed = INFERENCE_SEED
preprocessor = PolicyProcessorPipeline.from_pretrained(local_model, config_filename='policy_preprocessor.json')
postprocessor = PolicyProcessorPipeline.from_pretrained(local_model, config_filename='policy_postprocessor.json')
load_s = time.perf_counter() - t0
print(f'load={load_s:.1f}s, allocated={torch.cuda.memory_allocated()/2**30:.2f}GiB, peak={torch.cuda.max_memory_allocated()/2**30:.2f}GiB')

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation


def parc_obs_to_fastwam(obs, instruction):
    quat_xyzw = np.asarray(
        obs["robot0_eef_quat"],
        dtype=np.float32,
    )
    axis_angle = Rotation.from_quat(
        quat_xyzw
    ).as_rotvec().astype(np.float32)

    state = np.concatenate([
        np.asarray(
            obs["robot0_eef_pos"],
            dtype=np.float32,
        ),
        axis_angle,
        np.asarray(
            obs["robot0_gripper_qpos"],
            dtype=np.float32,
        ),
    ])
    assert state.shape == (8,)

    def image_chw01(key):
        image = np.asarray(obs[key], dtype=np.uint8)
        assert image.ndim == 3
        assert image.shape[-1] == 3

        chw = np.ascontiguousarray(
            image.transpose(2, 0, 1),
            dtype=np.float32,
        )
        chw = chw / 255.0
        return chw[None, ...]

    return {
        "observation.images.image": image_chw01(
            "agentview_image"
        ),
        "observation.images.image2": image_chw01(
            "robot0_eye_in_hand_image"
        ),
        "observation.state": state[None, ...],
        "task": instruction,
    }


dummy_obs = {
    "agentview_image": np.zeros(
        (128, 128, 3), dtype=np.uint8
    ),
    "robot0_eye_in_hand_image": np.zeros(
        (128, 128, 3), dtype=np.uint8
    ),
    "robot0_joint_pos": np.zeros(
        7, dtype=np.float32
    ),
    "robot0_eef_pos": np.array(
        [0.0, 0.0, 0.8], dtype=np.float32
    ),
    "robot0_eef_quat": np.array(
        [0.0, 0.0, 0.0, 1.0], dtype=np.float32
    ),
    "robot0_gripper_qpos": np.zeros(
        2, dtype=np.float32
    ),
}

raw_batch = parc_obs_to_fastwam(
    dummy_obs,
    "pick up the black bowl and place it on the plate",
)

print({
    key: value.shape if hasattr(value, "shape") else value
    for key, value in raw_batch.items()
})

In [ ]:
# 7. 単発推論と10秒ゲート。初回warmupと定常推論を分離します。
def infer_one(raw_batch, reset=False):
    if reset:
        policy.reset()

    processed = preprocessor(raw_batch)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        action = policy.select_action(processed)
        action = postprocessor({
            "action": action,
        })["action"]

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    action = (
        action.detach()
        .cpu()
        .numpy()
        .reshape(-1)
        .astype(np.float32)
    )

    assert action.shape == (7,)
    assert np.isfinite(action).all()

    return action, elapsed

policy.reset(); torch.cuda.reset_peak_memory_stats()
a0, warmup_s = infer_one(raw_batch, reset=True)
# queue消費ではなく再計画時間を測るため毎回reset
timings, actions = [], []
for seed in [41, 42, 43]:
    policy.config.inference_seed = seed
    a, dt = infer_one(raw_batch, reset=True)
    actions.append(a); timings.append(dt)
peak_gib = torch.cuda.max_memory_allocated()/2**30
print({'warmup_s': warmup_s, 'steady_s': timings, 'peak_gib': peak_gib, 'actions': np.stack(actions).tolist()})
assert max(timings) < 10.0, 'GATE FAIL: get_action 10秒制約を超過'
assert peak_gib < 23.5, 'GATE FAIL: L4 24GBで余裕がありません'

In [ ]:
policy.config.inference_seed = 42

a1, _ = infer_one(raw_batch, reset=True)
a2, _ = infer_one(raw_batch, reset=True)

print(a1)
print(a2)
print("max difference:", np.max(np.abs(a1 - a2)))

In [ ]:
from pathlib import Path


def directory_size_gib(path):
    path = Path(path)
    return sum(
        f.stat().st_size
        for f in path.rglob("*")
        if f.is_file()
    ) / 2**30


for path in [
    "/content/parc2026_fastwam/model_weights",
    "/content/hf_cache",
    "/root/.cache/huggingface",
]:
    path = Path(path)
    if path.exists():
        print(path, f"{directory_size_gib(path):.2f} GiB")

print(
    "current allocated:",
    torch.cuda.memory_allocated() / 2**30,
)
print(
    "current reserved:",
    torch.cuda.memory_reserved() / 2**30,
)
print(
    "peak allocated:",
    torch.cuda.max_memory_allocated() / 2**30,
)
print(
    "peak reserved:",
    torch.cuda.max_memory_reserved() / 2**30,
)

### データセット側フレーム → FastWAM入力

ゲートとNPZ生成が共用する。**提出用と同じ経路**（`preprocessor` → `select_action` →
`postprocessor`）を通すため、`infer_with_context` へ委譲するだけにする。

`raw_batch` の形は `parc_obs_to_fastwam` の戻り値に合わせる:
`observation.images.image` / `observation.images.image2` / `observation.state` / `task`


In [ ]:
# データセットのフレームを FastWAM の raw_batch へ変換するヘルパ。
# ⚠️ 提出用 MyPolicy と同じ経路を通す。ズレると「測ったもの」と「出すもの」が別になる。
import numpy as np
import torch

ORIENTATIONS = {
    "as_is":  lambda a: a,
    "flip_v": lambda a: a[::-1, :, :],
    "flip_h": lambda a: a[:, ::-1, :],
    "rot180": lambda a: a[::-1, ::-1, :],
}


def _as_hwc_uint8(v):
    """データセットの画像を HWC uint8 に揃える。CHW/float どちらで来ても通す。"""
    if isinstance(v, torch.Tensor):
        v = v.detach().cpu().numpy()
    v = np.asarray(v)
    if v.ndim == 4 and v.shape[0] == 1:
        v = v[0]
    if v.ndim == 3 and v.shape[0] in (1, 3) and v.shape[-1] not in (1, 3):
        v = np.transpose(v, (1, 2, 0))           # CHW -> HWC
    if v.dtype != np.uint8:
        v = (np.clip(v, 0.0, 1.0) * 255.0).astype(np.uint8)
    assert v.ndim == 3 and v.shape[-1] == 3, f"想定外の画像形状: {v.shape}"
    return v


def _chw01(hwc_uint8):
    chw = np.ascontiguousarray(hwc_uint8.transpose(2, 0, 1), dtype=np.float32) / 255.0
    return chw[None, ...]                        # (1,3,H,W)


def frame_to_raw_batch(front_hwc, wrist_hwc, state8, instruction, orient="as_is"):
    """parc_obs_to_fastwam と同じ形の dict を返す。"""
    fn = ORIENTATIONS[orient]
    state8 = np.asarray(state8, dtype=np.float32).reshape(-1)
    assert state8.shape == (8,), f"state は (8,) のはず: {state8.shape}"
    return {
        "observation.images.image":  _chw01(fn(front_hwc)),
        "observation.images.image2": _chw01(fn(wrist_hwc)),
        "observation.state":         state8[None, ...],
        "task":                      instruction,
    }


_CTX_CACHE = {}

def context_for(instruction: str):
    """instruction ごとに1回だけ text encoding（提出時の reset(instruction) と同じ扱い）。"""
    if instruction not in _CTX_CACHE:
        _CTX_CACHE[instruction] = encode_prompt_on_cpu(instruction)
    return _CTX_CACHE[instruction]


def base_action_for(raw_batch, instruction):
    ctx, ctx_mask = context_for(instruction)
    action, _ = infer_with_context(raw_batch, ctx, ctx_mask)   # (7,) numpy
    return np.asarray(action, dtype=np.float32).reshape(-1)


## 実データゲート & NPZ生成（adapter学習の前に必ず実行）

⚠️ ここまでの測定は**すべてダミー画像**（`dummy_obs` = 全ゼロ）。実画像で1フレームも通していない。

予選では画像の向きを間違えて**4タスクすべて成功率0**になった。しかも
`arr[::-1]`（上下だけ）で直したつもりが左右が鏡像のままで、**目視でも気づけなかった**。
ダミー画像ではこの種の不具合は原理的に検出できず、その上で学習を回しても
損失だけ下がって成功率は動かない（予選の教訓⑦「損失とスコアは別の量」）。

さらに `adapter_train.npz` の `base_action` は FastWAM の実出力なので、
**実データを通さない限り学習データそのものが作れない**。
ゲートとNPZ生成を同じ1パスで行う。MuJoCo/robosuite は不要。

判定は「向きを決め打ちしない」。4通り流して demo action との一致で選ぶ。


In [ ]:
# =====================================================================
# Colab の実行中セッションに、新しいセルとして「そのまま貼る」コード。
# 前提: infer_with_context / encode_prompt_on_cpu / preprocessor /
#       postprocessor / policy / device / RESULT_ROOT が定義済み
#       （ノートブックを上から流していれば揃っている）
#
# やること:
#   1) 配線ゲート … FastWAM の生出力が demo action と一致するか。
#                   画像の向き4通りを実測で比較する（決め打ちしない）
#   2) NPZ生成   … adapter_train.npz (state[N,8], base_action[N,7], target_action[N,7])
#
# MuJoCo/robosuite は不要。データセットのフレームを流すだけ。
# =====================================================================
import json, os, time
import numpy as np
import torch

# --- 429 対策。エラーは xet の CAS クライアントで出ていた -------------------
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_REPO     = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"

GATE_EPISODES = 2      # 向き判定はこれで足りる
GATE_FRAMES   = 24     # 約0.86秒/推論 × 4向き × 2話 × 24 ≒ 3分
NPZ_EPISODES  = 8      # 落とす動画チャンク量に直結
NPZ_STRIDE    = 2

# =====================================================================
# 1. フレーム → FastWAM 入力
#    ⚠️ parc_obs_to_fastwam と同じ形にする。提出時と同じ経路を通すため。
# =====================================================================
ORIENTATIONS = {
    "as_is":  lambda a: a,
    "flip_v": lambda a: a[::-1, :, :],
    "flip_h": lambda a: a[:, ::-1, :],
    "rot180": lambda a: a[::-1, ::-1, :],
}


def _as_hwc_uint8(v):
    """CHW/HWC・float/uint8 のどれで来ても HWC uint8 に揃える。"""
    if isinstance(v, torch.Tensor):
        v = v.detach().cpu().numpy()
    v = np.asarray(v)
    if v.ndim == 4 and v.shape[0] == 1:
        v = v[0]
    if v.ndim == 3 and v.shape[0] in (1, 3) and v.shape[-1] not in (1, 3):
        v = np.transpose(v, (1, 2, 0))
    if v.dtype != np.uint8:
        v = (np.clip(v, 0.0, 1.0) * 255.0).astype(np.uint8)
    assert v.ndim == 3 and v.shape[-1] == 3, f"想定外の画像形状: {v.shape}"
    return v


def _chw01(hwc):
    return (np.ascontiguousarray(hwc.transpose(2, 0, 1), dtype=np.float32) / 255.0)[None, ...]


def frame_to_raw_batch(front_hwc, wrist_hwc, state8, instruction, orient="as_is"):
    fn = ORIENTATIONS[orient]
    state8 = np.asarray(state8, dtype=np.float32).reshape(-1)
    assert state8.shape == (8,), f"state は (8,) のはず: {state8.shape}"
    return {
        "observation.images.image":  _chw01(fn(front_hwc)),
        "observation.images.image2": _chw01(fn(wrist_hwc)),
        "observation.state":         state8[None, ...],
        "task":                      instruction,
    }


_CTX_CACHE = {}

def context_for(instruction):
    if instruction not in _CTX_CACHE:
        _CTX_CACHE[instruction] = encode_prompt_on_cpu(instruction)
    return _CTX_CACHE[instruction]


def base_action_for(raw_batch, instruction):
    ctx, ctx_mask = context_for(instruction)
    action, _ = infer_with_context(raw_batch, ctx, ctx_mask)
    return np.asarray(action, dtype=np.float32).reshape(-1)


# =====================================================================
# 2. データセット（必要な話だけ・連続ブロック）
# =====================================================================
N_NEEDED = max(GATE_EPISODES, NPZ_EPISODES)
EPISODE_BLOCK = list(range(N_NEEDED))   # 連続ブロック = 落とすチャンク最小


def _load_dataset(retries=5):
    delay = 20
    for attempt in range(1, retries + 1):
        try:
            return LeRobotDataset(DATASET_REPO,
                                  revision=DATASET_REVISION,
                                  episodes=EPISODE_BLOCK)   # ← 無いと15.8GB全部引く
        except Exception as e:
            if attempt == retries or not ("429" in str(e) or "Too Many Requests" in str(e)):
                raise
            print(f"  429 ({attempt}/{retries})。{delay}秒待って再試行")
            time.sleep(delay); delay = min(delay * 2, 300)


print(f"dataset 読み込み（{N_NEEDED}話 / xet無効）…")
ds = _load_dataset()
print("frames:", ds.num_frames, " episodes:", ds.num_episodes)

_s0 = ds[0]
img_keys = [k for k in _s0 if k.startswith("observation.image")]
print("画像キー:", img_keys)
print("全キー:", sorted(_s0.keys()))
assert len(img_keys) >= 2, f"画像キーが2本無い: {img_keys}"


def _is_wrist(k):
    kl = k.lower()
    return ("wrist" in kl) or ("eye_in_hand" in kl) or kl.endswith("image2")


WRIST_KEY = next((k for k in img_keys if _is_wrist(k)), img_keys[1])
FRONT_KEY = next(k for k in img_keys if k != WRIST_KEY)
print(f"front={FRONT_KEY}  wrist={WRIST_KEY}")
print("⚠️ これが逆だと成功率0になる。下の1フレーム目の画像でも目視確認すること。")

STATE_KEY, ACTION_KEY = "observation.state", "action"


def _episode_range(ep):
    edi = getattr(ds, "episode_data_index", None)
    if edi is not None:
        return int(edi["from"][ep]), int(edi["to"][ep])
    lengths = [ds.meta.episodes[i]["length"] for i in range(ds.num_episodes)]
    lo = sum(lengths[:ep])
    return lo, lo + lengths[ep]


def _instruction(s):
    v = s.get("task", s.get("language_instruction", ""))
    return str(v[0] if isinstance(v, (list, tuple)) else v)


# 目視用に1フレーム保存
try:
    import matplotlib.pyplot as plt
    f0 = _as_hwc_uint8(_s0[FRONT_KEY]); w0 = _as_hwc_uint8(_s0[WRIST_KEY])
    fig, ax = plt.subplots(1, 2, figsize=(6, 3))
    ax[0].imshow(f0); ax[0].set_title(f"front\n{FRONT_KEY}", fontsize=7); ax[0].axis("off")
    ax[1].imshow(w0); ax[1].set_title(f"wrist\n{WRIST_KEY}", fontsize=7); ax[1].axis("off")
    plt.tight_layout(); plt.show()
    print("instruction:", _instruction(_s0))
except Exception as e:
    print("(画像表示スキップ:", e, ")")


# =====================================================================
# 3. ゲート判定
# =====================================================================
def run_pass(episodes, max_frames, stride, orient):
    S, B, T = [], [], []
    for ep in episodes:
        lo, hi = _episode_range(ep)
        for i in range(lo, min(hi, lo + max_frames * stride), stride):
            s = ds[i]
            st = s[STATE_KEY]
            st = st.detach().cpu().numpy() if isinstance(st, torch.Tensor) else np.asarray(st)
            ac = s[ACTION_KEY]
            ac = ac.detach().cpu().numpy() if isinstance(ac, torch.Tensor) else np.asarray(ac)
            ins = _instruction(s)
            rb = frame_to_raw_batch(_as_hwc_uint8(s[FRONT_KEY]),
                                    _as_hwc_uint8(s[WRIST_KEY]), st, ins, orient)
            S.append(np.asarray(st, np.float32).reshape(-1))
            B.append(base_action_for(rb, ins))
            T.append(np.asarray(ac, np.float32).reshape(-1))
    return np.stack(S), np.stack(B), np.stack(T)


def agreement(base, target):
    bd, td = base - base.mean(0), target - target.mean(0)
    den = np.linalg.norm(bd, axis=0) * np.linalg.norm(td, axis=0) + 1e-8
    pd = (bd * td).sum(0) / den
    return {"mse": float(((base - target) ** 2).mean()),
            "corr_xyz": float(np.mean(pd[:3])),
            "corr_rot": float(np.mean(pd[3:6])),
            "corr_grip": float(pd[6]),
            "per_dim": [float(x) for x in pd]}


print(f"\n配線ゲート: 4向き × {GATE_EPISODES}話 × {GATE_FRAMES}フレーム")
report = {}
for orient in ORIENTATIONS:
    t0 = time.perf_counter()
    S, B, T = run_pass(range(GATE_EPISODES), GATE_FRAMES, 1, orient)
    m = agreement(B, T); m["seconds"] = round(time.perf_counter() - t0, 1)
    report[orient] = m
    print(f"  {orient:7s} corr_xyz={m['corr_xyz']:+.3f} corr_rot={m['corr_rot']:+.3f} "
          f"mse={m['mse']:.4f} ({m['seconds']}s)")

best = max(report, key=lambda k: report[k]["corr_xyz"])
spread = report[best]["corr_xyz"] - min(r["corr_xyz"] for r in report.values())
GATE_PASS = report[best]["corr_xyz"] > 0.30

print(f"\n最良: {best}  corr_xyz={report[best]['corr_xyz']:+.3f}  向き間の差={spread:.3f}")
print("=== 判定 ===")
if not GATE_PASS:
    print("❌ FAIL: どの向きでも demo と相関しない。向き以外の配線を疑う。")
    print("   候補: state の並び / 正規化統計 / instruction / front-wrist の取り違え")
    print("   ⚠️ この状態で adapter 学習へ進まない。損失は下がるが成功率は動かない。")
elif spread < 0.10:
    print("⚠️ WARN: 向きを変えても結果がほぼ同じ = 画像が効いていない可能性。")
else:
    print(f"✅ PASS: {best} で明確に一致。")

(RESULT_ROOT / "gate_report.json").write_text(
    json.dumps({"orientations": report, "best": best, "pass": bool(GATE_PASS),
                "spread": float(spread), "front_key": FRONT_KEY,
                "wrist_key": WRIST_KEY}, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", RESULT_ROOT / "gate_report.json")

# =====================================================================
# 4. NPZ生成（ゲート通過時のみ）
# =====================================================================
if GATE_PASS:
    print(f"\nNPZ生成: {NPZ_EPISODES}話 / stride {NPZ_STRIDE} / 向き {best}")
    S, B, T = run_pass(range(min(NPZ_EPISODES, ds.num_episodes)), 10**6, NPZ_STRIDE, best)
    np.savez(RESULT_ROOT / "adapter_train.npz",
             state=S, base_action=B, target_action=T)
    resid = float(np.abs(T - B).mean())
    print(f"  N={len(S)} -> {RESULT_ROOT/'adapter_train.npz'}")
    print(f"  |target-base| 平均 = {resid:.4f}")
    print(f"  ⚠️ 提出用 MyPolicy に反映する向き: {best}")
    if resid < 0.01:
        print("  ⚠️ residual が極端に小さい。adapter がほぼ何も学ばず、")
        print("     独自学習要件を満たさない恐れ（予選の『24件中23件が同一』と同じ形）。")
else:
    print("\nNPZは作らない。配線を直してから再実行する。")


In [ ]:
# =====================================================================
# 追加セルとして貼る（前のセルの ds / frame_to_raw_batch などはそのまま使う）
#
# 直した点:
#   encode_prompt_on_cpu は前のランタイムで定義したもので、今のセッションには無い。
#   A100 80GB では CPU 退避は不要なので、通常経路 infer_one を使う。
#   raw_batch の "task" キーからモデル内部でテキストが処理される。
#
# ⚠️ policy.select_action は内部に action キューを持つ。
#    reset=True にしないと2回目以降が前回予測の使い回しになり、ゲートが無意味になる。
# =====================================================================
import numpy as np, torch, json, time

_HAS_CPU_CTX = "encode_prompt_on_cpu" in globals()
print("text encoder 経路:", "CPU offload (encode_prompt_on_cpu)" if _HAS_CPU_CTX
      else "通常 (infer_one / task キー)")
assert _HAS_CPU_CTX or ("infer_one" in globals()), \
    "infer_one も encode_prompt_on_cpu も無い。モデルロードのセルを先に実行すること"


def base_action_for(raw_batch, instruction):
    """FastWAM の action chunk 先頭1本。毎回 reset して再計画させる。"""
    if _HAS_CPU_CTX:
        ctx, m = context_for(instruction)
        action, _ = infer_with_context(raw_batch, ctx, m)     # 中で reset 済み
    else:
        action, _ = infer_one(raw_batch, reset=True)          # ← キュー使い回しを防ぐ
    return np.asarray(action, dtype=np.float32).reshape(-1)


# --- 健全性チェック: 同じ入力で2回呼んで一致するか（seed固定なので一致すべき） ---
_s = ds[0]
_st = _s["observation.state"]
_st = _st.detach().cpu().numpy() if isinstance(_st, torch.Tensor) else np.asarray(_st)
_ins = _instruction(_s)
_rb = frame_to_raw_batch(_as_hwc_uint8(_s[FRONT_KEY]), _as_hwc_uint8(_s[WRIST_KEY]),
                         _st, _ins, "as_is")
_a1 = base_action_for(_rb, _ins)
_a2 = base_action_for(_rb, _ins)
print("同一入力2回の差:", float(np.abs(_a1 - _a2).max()), " (seed固定なら0付近)")
print("action例:", np.round(_a1, 4))

# =====================================================================
# ゲート再実行
# =====================================================================
print(f"\n配線ゲート: 4向き × {GATE_EPISODES}話 × {GATE_FRAMES}フレーム")
report = {}
for orient in ORIENTATIONS:
    t0 = time.perf_counter()
    S, B, T = run_pass(range(GATE_EPISODES), GATE_FRAMES, 1, orient)
    m = agreement(B, T); m["seconds"] = round(time.perf_counter() - t0, 1)
    report[orient] = m
    print(f"  {orient:7s} corr_xyz={m['corr_xyz']:+.3f} corr_rot={m['corr_rot']:+.3f} "
          f"mse={m['mse']:.4f} ({m['seconds']}s)")

best = max(report, key=lambda k: report[k]["corr_xyz"])
spread = report[best]["corr_xyz"] - min(r["corr_xyz"] for r in report.values())
GATE_PASS = report[best]["corr_xyz"] > 0.30

print(f"\n最良: {best}  corr_xyz={report[best]['corr_xyz']:+.3f}  向き間の差={spread:.3f}")
print("次元別相関:", np.round(report[best]["per_dim"], 3))
print("=== 判定 ===")
if not GATE_PASS:
    print("❌ FAIL: どの向きでも demo と相関しない。向き以外の配線を疑う。")
    print("   候補: state の並び / 正規化統計 / instruction / front-wrist の取り違え")
    print("   ⚠️ この状態で adapter 学習へ進まない。損失は下がるが成功率は動かない。")
elif spread < 0.10:
    print("⚠️ WARN: 向きを変えても結果がほぼ同じ = 画像が効いていない可能性。")
else:
    print(f"✅ PASS: {best} で明確に一致。")

(RESULT_ROOT / "gate_report.json").write_text(
    json.dumps({"orientations": report, "best": best, "pass": bool(GATE_PASS),
                "spread": float(spread), "front_key": FRONT_KEY,
                "wrist_key": WRIST_KEY,
                "text_path": "cpu_ctx" if _HAS_CPU_CTX else "infer_one"},
               ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", RESULT_ROOT / "gate_report.json")

# =====================================================================
# NPZ生成（ゲート通過時のみ）
# =====================================================================
if GATE_PASS:
    print(f"\nNPZ生成: {min(NPZ_EPISODES, ds.num_episodes)}話 / stride {NPZ_STRIDE} / 向き {best}")
    S, B, T = run_pass(range(min(NPZ_EPISODES, ds.num_episodes)), 10**6, NPZ_STRIDE, best)
    np.savez(RESULT_ROOT / "adapter_train.npz",
             state=S, base_action=B, target_action=T)
    resid = float(np.abs(T - B).mean())
    print(f"  N={len(S)} -> {RESULT_ROOT/'adapter_train.npz'}")
    print(f"  |target-base| 平均 = {resid:.4f}")
    print(f"  ⚠️ 提出用 MyPolicy に反映する向き: {best}")
    if resid < 0.01:
        print("  ⚠️ residual が極端に小さい。adapter がほぼ何も学ばず")
        print("     独自学習要件を満たさない恐れ（予選の『24件中23件が同一』と同じ形）。")
else:
    print("\nNPZは作らない。配線を直してから再実行する。")


## 小型residual adapter（自前学習要素）

FastWAMの先頭Actionと8次元stateを入力し、7次元補正を学習します。これは接続確認用の最小案です。最終提出では、補正量・成功率への寄与を記録し、規約上『実質的に寄与』すると説明できる場合だけ採用します。`adapter_train.npz`には `state [N,8]`, `base_action [N,7]`, `target_action [N,7]` を入れます。

In [ ]:
# =====================================================================
# residual adapter 学習（検証分割つき）。既存の学習セルの代わりに貼る。
#
# なぜ差し替えるか:
#   元セルは 579サンプルに対し約19,463パラメータ（約5倍の過剰）を
#   ミニバッチ無し・1000ステップで回し、検証分割が無い。
#   損失は必ず下がるが、暗記か汎化かを区別できない。
#   → 予選の教訓⑦「損失とスコアは別の量」がそのまま効く場所。
#
# 分割の仕方:
#   run_pass はエピソード順にサンプルを並べるので、末尾を切れば
#   「学習に使っていないエピソード」の検証集合になる（時間相関のリークを避ける）。
# =====================================================================
import json, time
import numpy as np
import torch
import torch.nn as nn

ADAPTER_DATA  = RESULT_ROOT / "adapter_train.npz"
ADAPTER_SCALE = 0.20
VAL_FRACTION  = 0.25          # 末尾25% ≒ 最後の2エピソード
STEPS         = 1000
BATCH         = 128           # ミニバッチを切る（元セルは全バッチだった）
LR            = 3e-4
WD            = 1e-2          # 過剰パラメータなので元の1e-4より強めに

d = np.load(ADAPTER_DATA)
S_all = torch.tensor(d["state"], dtype=torch.float32, device=device)
B_all = torch.tensor(d["base_action"], dtype=torch.float32, device=device)
T_all = torch.tensor(d["target_action"], dtype=torch.float32, device=device)
N = len(S_all)
n_val = int(N * VAL_FRACTION)
n_tr = N - n_val

S, B, T = S_all[:n_tr], B_all[:n_tr], T_all[:n_tr]
Sv, Bv, Tv = S_all[n_tr:], B_all[n_tr:], T_all[n_tr:]
print(f"N={N}  train={n_tr}  val={n_val}（末尾＝学習に使わないエピソード）")

DIM = ["x", "y", "z", "rx", "ry", "yaw", "grip"]


def per_dim_mse(pred, target):
    return ((pred - target) ** 2).mean(0).detach().cpu().numpy()


base_val = per_dim_mse(Bv, Tv)
print("\n補正なしの val MSE（次元別）:")
print("   " + "  ".join(f"{n}={v:.4f}" for n, v in zip(DIM, base_val)))
print(f"   合計 {base_val.mean():.5f}")


class ResidualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(15, 128), nn.SiLU(),
                                 nn.Linear(128, 128), nn.SiLU(),
                                 nn.Linear(128, 7))

    def forward(self, state, base_action):
        return ADAPTER_SCALE * torch.tanh(self.net(torch.cat([state, base_action], dim=-1)))


torch.manual_seed(42)
adapter = ResidualAdapter().to(device)
n_param = sum(p.numel() for p in adapter.parameters())
print(f"\nパラメータ数 {n_param:,} / 学習サンプル {n_tr}（拘束 {n_tr*7:,}）")

opt = torch.optim.AdamW(adapter.parameters(), lr=LR, weight_decay=WD)
hist, best_val, best_state, best_step = [], float("inf"), None, 0

for step in range(1, STEPS + 1):
    idx = torch.randint(0, n_tr, (min(BATCH, n_tr),), device=device)
    pred = torch.clamp(B[idx] + adapter(S[idx], B[idx]), -1, 1)
    loss = ((pred - T[idx]) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

    if step % 25 == 0:
        adapter.eval()
        with torch.no_grad():
            pv = torch.clamp(Bv + adapter(Sv, Bv), -1, 1)
            vl = float(((pv - Tv) ** 2).mean())
        adapter.train()
        hist.append({"step": step, "train": float(loss), "val": vl})
        if vl < best_val:
            best_val, best_step = vl, step
            best_state = {k: v.detach().clone() for k, v in adapter.state_dict().items()}
        if step % 100 == 0:
            print(f"  step {step:4d}  train={float(loss):.5f}  val={vl:.5f}"
                  + ("  ← best" if step == best_step else ""))

print(f"\n最良 val = {best_val:.5f} (step {best_step})")
adapter.load_state_dict(best_state)   # 早期停止の重みへ戻す
adapter.eval()

# =====================================================================
# 判定
# =====================================================================
with torch.no_grad():
    pv = torch.clamp(Bv + adapter(Sv, Bv), -1, 1)
    after_val = per_dim_mse(pv, Tv)
    delta = adapter(Sv, Bv)
    mag = delta.abs().mean(0).detach().cpu().numpy()

print("\n次元別 val MSE（補正なし → 補正あり）:")
for n, b, a, m in zip(DIM, base_val, after_val, mag):
    arrow = "改善" if a < b * 0.98 else ("悪化" if a > b * 1.02 else "変化なし")
    print(f"   {n:4s} {b:.5f} -> {a:.5f}   補正量 {m:.4f}   {arrow}")

impr = (base_val.mean() - after_val.mean()) / base_val.mean() * 100
print(f"\n合計 {base_val.mean():.5f} -> {after_val.mean():.5f}   ({impr:+.1f}%)")

print("\n=== 判定 ===")
if after_val.mean() >= base_val.mean():
    print("❌ val が改善していない = 暗記。このまま提出しても意味が無い。")
    print("   ADAPTER_SCALE を下げる / データを増やす / そもそも adapter を使わない。")
elif impr < 3:
    print("⚠️ 改善が小さい。独自学習要件は形式的には満たすが、")
    print("   『実質的に寄与』の説明が弱い。補正量とあわせて記録しておく。")
else:
    print(f"✅ val で {impr:.1f}% 改善。学習が汎化している。")

weak = [n for n, m in zip(DIM, mag) if m > 0.01]
print(f"\n補正が集中している次元: {weak}")
print("   ゲートで弱かったのは z(0.125) と yaw(0.047)。")
print("   そこに補正が向いていれば筋が良い。x,y を大きく触っていたら過学習を疑う。")

# =====================================================================
# 保存（private HF へも退避）
# =====================================================================
out = RESULT_ROOT / "residual_adapter.pt"
torch.save({"state_dict": adapter.state_dict(), "scale": ADAPTER_SCALE,
            "run_id": RUN_ID, "best_step": best_step, "best_val": best_val,
            "orientation": "as_is",
            "val_mse_before": base_val.tolist(), "val_mse_after": after_val.tolist(),
            "correction_magnitude": mag.tolist()}, out)
(RESULT_ROOT / "adapter_history.json").write_text(
    json.dumps(hist, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nsaved:", out)

try:
    hf_api.upload_file(path_or_fileobj=str(out),
                       path_in_repo=f"{RUN_ID}/residual_adapter.pt",
                       repo_id=HF_BACKUP_REPO,
                       commit_message=f"{RUN_ID}: adapter val={best_val:.5f}")
    print("private HF backup: COMPLETE")
except Exception as e:
    print("⚠️ HF退避に失敗:", e)
    print("   ランタイムを切る前に必ず手元へ落とすこと。")


In [ ]:
# 9. 実験manifest保存
manifest = {
    **runtime, 'run_id': RUN_ID, 'model_id': MODEL_ID,
    'inference_steps': INFERENCE_STEPS, 'n_action_steps': N_ACTION_STEPS,
    'model_snapshot_gib': total_gib, 'load_seconds': load_s,
    'warmup_seconds': warmup_s, 'steady_seconds': timings,
    'peak_vram_gib': peak_gib, 'adapter_data_exists': ADAPTER_DATA.exists(),
}
manifest_path = RESULT_ROOT/'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
hf_api.upload_file(path_or_fileobj=str(manifest_path), path_in_repo=f'{RUN_ID}/run_manifest.json', repo_id=HF_BACKUP_REPO, commit_message=f'{RUN_ID}: manifest')
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
for f in ["gate_report.json", "adapter_history.json", "manifest.json"]:
    p = RESULT_ROOT / f
    if p.exists():
        hf_api.upload_file(path_or_fileobj=str(p), path_in_repo=f"{RUN_ID}/{f}",
                           repo_id=HF_BACKUP_REPO, commit_message=f"{RUN_ID}: {f}")
        print("uploaded:", f)
print("private HF backup: COMPLETE")


## 次の判定

- ロード不可 / 24GB超過 / 10秒超過 / 必須資産込み20GB超過: 明日のWAM提出は中止。
- 単発推論PASS: PARC公式リポジトリと評価データをColabへ配置し、1タスク×1 episodeへ進む。
- 1 episodeで制御成立: 4タスク×3 seedとadapter学習前後比較を行う。
- 公開重み単体は提出しない。自前学習要素がActionと成功率へ寄与した証拠を残す。

In [ ]:
import gc
import torch


def encode_prompt_on_cpu(instruction):
    prompt = policy.config.prompt_template.format(
        task=instruction
    )

    text_encoder = policy.model.text_encoder
    tokenizer = policy.model.tokenizer

    ids, mask = tokenizer(
        prompt,
        return_mask=True,
        add_special_tokens=True,
    )

    ids = ids.cpu()
    mask = mask.cpu().bool()

    with torch.inference_mode():
        context = text_encoder(ids, mask)

        # padding部分を非破壊演算でゼロ化する
        context = context * mask.unsqueeze(-1).to(
            dtype=context.dtype
        )

        context_mask = torch.ones_like(
            mask,
            dtype=torch.bool,
        )

    return (
        context.to("cuda"),
        context_mask.to("cuda"),
    )


# Text encoderをCPUへ退避
policy.model.text_encoder.cpu()

gc.collect()
torch.cuda.empty_cache()

print(
    "offload後 allocated:",
    torch.cuda.memory_allocated() / 2**30,
)
print(
    "offload後 reserved:",
    torch.cuda.memory_reserved() / 2**30,
)

In [ ]:
instruction = (
    "pick up the black bowl "
    "and place it on the plate"
)

t0 = time.perf_counter()

context, context_mask = encode_prompt_on_cpu(
    instruction
)

print(
    "CPU text encoding:",
    time.perf_counter() - t0,
    "seconds",
)
print(context.shape, context_mask.shape)

In [ ]:
def infer_with_context(
    raw_batch,
    context,
    context_mask,
):
    policy.reset()

    processed = preprocessor(raw_batch)

    # promptとcontextは排他的
    processed.pop("task", None)
    processed.pop("prompt", None)

    processed["context"] = context
    processed["context_mask"] = context_mask

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    with torch.inference_mode():
        action = policy.select_action(processed)
        action = postprocessor({
            "action": action,
        })["action"]

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    action = (
        action.detach()
        .cpu()
        .numpy()
        .reshape(-1)
        .astype(np.float32)
    )

    return action, elapsed


policy.config.inference_seed = 42

action, elapsed = infer_with_context(
    raw_batch,
    context,
    context_mask,
)

print("action:", action)
print("elapsed:", elapsed)
print(
    "peak allocated:",
    torch.cuda.max_memory_allocated() / 2**30,
)
print(
    "peak reserved:",
    torch.cuda.max_memory_reserved() / 2**30,
)

In [ ]:
from pathlib import Path

cache_root = Path(
    "/root/.cache/huggingface/hub"
)

physical_files = {}

for file in cache_root.rglob("*"):
    if not file.is_file():
        continue

    real = file.resolve()

    if real not in physical_files:
        physical_files[real] = real.stat().st_size

total = sum(physical_files.values())

print(
    "physical unique total:",
    total / 2**30,
    "GiB",
)

for path, size in sorted(
    physical_files.items(),
    key=lambda item: item[1],
    reverse=True,
)[:30]:
    print(
        f"{size / 2**30:7.2f} GiB",
        path,
    )

In [ ]:
from pathlib import Path

hub = Path("/root/.cache/huggingface/hub")

for snapshot in hub.glob(
    "models--Wan-AI--Wan2.2-TI2V-5B-Diffusers/"
    "snapshots/*"
):
    for file in snapshot.rglob("*"):
        if not file.is_file():
            continue

        size = file.resolve().stat().st_size

        if size > 100 * 1024**2:
            print(
                f"{size / 2**30:6.2f} GiB",
                file.relative_to(snapshot),
            )

In [ ]:
import gc
import torch
import torch.nn as nn
import psutil

memory = psutil.virtual_memory()

print(
    "CPU RAM:",
    memory.available / 2**30,
    "GiB available",
)

assert memory.available > 30 * 2**30, (
    "量子化中の一時メモリが足りません"
)

In [ ]:
text_wrapper = policy.model.text_encoder

# dynamic INT8はfloat32モデルから作る
text_wrapper.model = text_wrapper.model.float()

gc.collect()

text_wrapper.model = (
    torch.ao.quantization.quantize_dynamic(
        text_wrapper.model,
        {nn.Linear},
        dtype=torch.qint8,
        inplace=True,
    )
)

gc.collect()

print("UMT5 INT8 conversion: PASS")

In [ ]:
t0 = time.perf_counter()

context_int8, mask_int8 = encode_prompt_on_cpu(
    instruction
)

print(
    "INT8 text encoding:",
    time.perf_counter() - t0,
    "seconds",
)
print(context_int8.shape)

In [ ]:
difference = (
    context_int8.float() - context.float()
).abs()

print("context mean error:", difference.mean().item())
print("context max error:", difference.max().item())

In [ ]:
policy.config.inference_seed = 42

action_int8, elapsed_int8 = infer_with_context(
    raw_batch,
    context_int8,
    mask_int8,
)

print("BF16 action:", action)
print("INT8 action :", action_int8)
print(
    "action max difference:",
    np.max(np.abs(action_int8 - action)),
)
print("inference:", elapsed_int8)

In [ ]:
import gc
import torch.nn as nn

from torch.ao.quantization import (
    float_qparams_weight_only_qconfig,
    quantize_dynamic,
)

# 現在残っている非量子化Embeddingを確認
embedding_before = sum(
    isinstance(module, nn.Embedding)
    for module in text_wrapper.model.modules()
)

print(
    "FP32 Embedding before:",
    embedding_before,
)

# Embeddingだけ追加でINT8 weight-only化
text_wrapper.model = quantize_dynamic(
    text_wrapper.model,
    qconfig_spec={
        nn.Embedding:
            float_qparams_weight_only_qconfig,
    },
    inplace=True,
)

gc.collect()

embedding_after = sum(
    isinstance(module, nn.Embedding)
    for module in text_wrapper.model.modules()
)

print(
    "FP32 Embedding after:",
    embedding_after,
)

assert embedding_before > 0
assert embedding_after == 0

In [ ]:
quantized_path = Path(
    "/content/umt5_int8_state.pt"
)

torch.save(
    text_wrapper.model.state_dict(),
    quantized_path,
)

size_gib = (
    quantized_path.stat().st_size / 2**30
)

print(
    "quantized UMT5:",
    size_gib,
    "GiB",
)

assert size_gib < 6.0, (
    "20GB提出上限へ収まりません"
)

In [ ]:
from collections import defaultdict
from pathlib import Path

from safetensors import safe_open

vae_files = list(
    Path(
        "/root/.cache/huggingface/hub/"
        "models--Wan-AI--Wan2.2-TI2V-5B-Diffusers/"
        "snapshots"
    ).rglob(
        "vae/diffusion_pytorch_model.safetensors"
    )
)

assert len(vae_files) == 1, vae_files
vae_file = vae_files[0]

groups = defaultdict(int)

with safe_open(
    vae_file,
    framework="pt",
    device="cpu",
) as tensors:
    for key in tensors.keys():
        tensor = tensors.get_slice(key)
        shape = tensor.get_shape()

        elements = 1
        for dimension in shape:
            elements *= dimension

        dtype = str(tensor.get_dtype())

        bytes_per_element = {
            "F64": 8,
            "F32": 4,
            "F16": 2,
            "BF16": 2,
            "I64": 8,
            "I32": 4,
            "I16": 2,
            "I8": 1,
            "U8": 1,
        }.get(dtype, 2)

        prefix = key.split(".", 1)[0]
        groups[prefix] += (
            elements * bytes_per_element
        )

for name, size in sorted(
    groups.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(
        f"{size / 2**30:7.3f} GiB",
        name,
    )

In [ ]:
import gc
import numpy as np
import torch

# decoder削除前の基準
policy.config.inference_seed = 42

action_before, _ = infer_with_context(
    raw_batch,
    context_int8,
    mask_int8,
)

print("before:", action_before)
print(
    "VRAM before:",
    torch.cuda.memory_allocated() / 2**30,
)

In [ ]:
vae_model = policy.model.vae.vae

assert vae_model.decoder is not None

vae_model.decoder = None

gc.collect()
torch.cuda.empty_cache()

print(
    "VRAM after decoder removal:",
    torch.cuda.memory_allocated() / 2**30,
)

In [ ]:
policy.config.inference_seed = 42

action_after, elapsed_after = infer_with_context(
    raw_batch,
    context_int8,
    mask_int8,
)

print("after :", action_after)
print(
    "max difference:",
    np.max(
        np.abs(action_after - action_before)
    ),
)
print("elapsed:", elapsed_after)

In [ ]:
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file
import shutil

original_vae_dir = vae_file.parent
trimmed_vae_dir = Path(
    "/content/vae_encoder_only"
)
trimmed_vae_dir.mkdir(
    parents=True,
    exist_ok=True,
)

trimmed_tensors = {}

with safe_open(
    vae_file,
    framework="pt",
    device="cpu",
) as source:
    for key in source.keys():
        if key.startswith("decoder."):
            continue

        trimmed_tensors[key] = (
            source.get_tensor(key)
        )

trimmed_weights = (
    trimmed_vae_dir
    / "diffusion_pytorch_model.safetensors"
)

save_file(
    trimmed_tensors,
    trimmed_weights,
)

shutil.copy2(
    original_vae_dir / "config.json",
    trimmed_vae_dir / "config.json",
)

print(
    "encoder-only VAE:",
    trimmed_weights.stat().st_size / 2**30,
    "GiB",
)

In [ ]:
import torch

from diffusers import AutoencoderKLWan
from safetensors.torch import load_file
from lerobot.policies.fastwam.wan.adapters import (
    WanVideoVAE38,
)

vae_config = AutoencoderKLWan.load_config(
    trimmed_vae_dir
)

rebuilt_vae = AutoencoderKLWan.from_config(
    vae_config
)

trimmed_state = load_file(
    trimmed_weights,
    device="cpu",
)

result = rebuilt_vae.load_state_dict(
    trimmed_state,
    strict=False,
)

unexpected = result.unexpected_keys
missing_non_decoder = [
    key
    for key in result.missing_keys
    if not key.startswith("decoder.")
]

print("unexpected:", unexpected)
print(
    "missing non-decoder:",
    missing_non_decoder,
)

assert unexpected == []
assert missing_non_decoder == []

rebuilt_vae.decoder = None

rebuilt_wrapper = WanVideoVAE38(
    dtype=torch.bfloat16,
    device="cuda",
    pretrained=rebuilt_vae,
)

print("encoder-only VAE rebuild: PASS")

In [ ]:
# 0.557GiBから再構築したVAEへ差し替える
object.__setattr__(
    policy.model,
    "vae",
    rebuilt_wrapper,
)

policy.config.inference_seed = 42

action_rebuilt, elapsed_rebuilt = (
    infer_with_context(
        raw_batch,
        context_int8,
        mask_int8,
    )
)

print("reference:", action_before)
print("rebuilt  :", action_rebuilt)

print(
    "max difference:",
    np.max(np.abs(
        action_rebuilt - action_before
    )),
)

print(
    "elapsed:",
    elapsed_rebuilt,
)

print(
    "VRAM:",
    torch.cuda.memory_allocated() / 2**30,
    "GiB",
)

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import torch

LIGHT_ROOT = Path(
    "/content/fastwam_light_assets"
)
TEXT_DIR = LIGHT_ROOT / "text_encoder_int8"
VAE_DIR = LIGHT_ROOT / "vae_encoder_only"
TOKENIZER_DIR = LIGHT_ROOT / "tokenizer"

TEXT_DIR.mkdir(parents=True, exist_ok=True)
VAE_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

# INT8 UMT5をモデルごと保存
text_model_path = (
    TEXT_DIR / "umt5_encoder_int8.pt"
)

torch.save(
    text_wrapper.model,
    text_model_path,
)

# tokenizer
policy.model.tokenizer.tokenizer.save_pretrained(
    TOKENIZER_DIR
)

# encoder-only VAE
shutil.copy2(
    trimmed_weights,
    VAE_DIR
    / "diffusion_pytorch_model.safetensors",
)

shutil.copy2(
    trimmed_vae_dir / "config.json",
    VAE_DIR / "config.json",
)


def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as stream:
        while block := stream.read(16 * 1024**2):
            digest.update(block)

    return digest.hexdigest()


manifest = {
    "run_id": RUN_ID,
    "base_model": MODEL_ID,
    "text_encoder": {
        "format": "torch_dynamic_int8",
        "size_gib":
            text_model_path.stat().st_size / 2**30,
        "sha256": sha256(text_model_path),
    },
    "vae": {
        "format": "encoder_only_safetensors",
        "size_gib":
            (
                VAE_DIR
                / "diffusion_pytorch_model.safetensors"
            ).stat().st_size / 2**30,
        "sha256": sha256(
            VAE_DIR
            / "diffusion_pytorch_model.safetensors"
        ),
    },
    "validated": {
        "same_seed_deterministic": True,
        "decoder_removal_action_max_diff": 0.0,
        "rebuilt_vae_action_max_diff": 0.0,
        "action_inference_seconds":
            elapsed_rebuilt,
        "runtime_vram_gib":
            torch.cuda.memory_allocated() / 2**30,
    },
}

manifest_path = LIGHT_ROOT / "manifest.json"
manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
))

In [ ]:
total = sum(
    file.stat().st_size
    for file in LIGHT_ROOT.rglob("*")
    if file.is_file()
)

print(
    "light assets:",
    total / 2**30,
    "GiB",
)

In [ ]:
hf_api.upload_folder(
    folder_path=str(LIGHT_ROOT),
    path_in_repo=(
        f"{RUN_ID}/light_assets"
    ),
    repo_id=HF_BACKUP_REPO,
    commit_message=(
        f"{RUN_ID}: validated INT8 UMT5 "
        "and encoder-only VAE"
    ),
)

print("private HF backup: COMPLETE")